# data_02_interp_eval  –  보간법 선정 근거 확보

## 목적
전처리 파이프라인(`data_01_pipeline`) 에서 사용할 결측치 보간법을 정량적으로 비교·선정한다.
이전 연구에서 보간법 선정 근거가 불명확하였던 점을 보완한다.

## 비교 대상 (3안)
| 안 | 방법 | 설명 |
|---|---|---|
| 기존안 | 행 평균 대치 + 선형 보간 | `fill_missing_wide()` 현행 방식 |
| 1안 | PCHIP 보간 | 단조성 보존, C¹ 연속 곡선 |
| 2안 | STL 분해 후 재합성 | 계절·추세 구조 보존 |

## 평가 지표 (4종)
| 지표 | 의미 | 가중치 |
|---|---|---|
| RMSE (Held-out) | 마스킹된 실제값과의 재현 오차 | 0.50 |
| Roughness | 2차 차분 기반 매끄러움 | 0.20 |
| ACF Divergence | 자기상관 구조 보존 여부 | 0.20 |
| Wasserstein Distance | 분포 보존 여부 | 0.10 |

## 01. Init

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = [14, 5]

from scipy.interpolate import PchipInterpolator
from scipy.stats import wasserstein_distance
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf
from sklearn.metrics import mean_squared_error

from core import provider_kier_m02 as com_KIER_M02

In [ ]:
## 도메인 설정
int_domain = 0   # 0=ELEC — 결측이 가장 많은 도메인으로 비교
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
_, _, _, _, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)

STL_PERIOD = 144    # 10분 × 144 = 24H (일주기)
N_TRIALS   = 10     # 마스킹 반복 횟수
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 02. 데이터 로드
실제 결측이 없는 완전한 세대(complete cases)를 실험 대상으로 사용한다.

In [ ]:
## 10분 단위 INST wide 데이터 로드 (보간 전 버전)
str_file = 'KIER_' + str_domain + '_INST_10MIN.csv'
df_raw = pd.read_csv(str_dirName_h + str_file, index_col=0)
df_raw['METER_DATE'] = pd.to_datetime(df_raw['METER_DATE'])
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
## 세대 컬럼 추출
META_COLS = {'METER_DATE','YEAR','MONTH','DAY','HOUR','MINUTE','MEAN_OF_INST','SUM_OF_INST'}
house_cols = [c for c in df_raw.columns if c not in META_COLS]
print(f'세대 수: {len(house_cols)}')

## 완전한 세대(결측 없음) 선별 → 실험 대상
complete_houses = [c for c in house_cols if df_raw[c].isna().sum() == 0]
print(f'완전한 세대 수: {len(complete_houses)}')

## 실험 샘플: 완전한 세대 중 20개 무작위 선택
sample_houses = pd.Series(complete_houses).sample(20, random_state=RANDOM_SEED).tolist()
df_eval = df_raw[['METER_DATE'] + sample_houses].copy()
print(f'실험 데이터 shape: {df_eval.shape}')

## 03. 보간 함수 정의

In [ ]:
# ── 기존안: 행 평균 대치 + 선형 보간 ─────────────────────────────
def interp_baseline(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Stage 1: 행 평균 대치 / Stage 2: 보간된 평균 대치 / Stage 3: 선형 보간"""
    df = df.copy()
    row_mean = df[cols].mean(axis=1)
    df[cols] = df[cols].apply(lambda c: c.fillna(row_mean))
    row_mean_interp = df[cols].mean(axis=1).interpolate(method='linear')
    df[cols] = df[cols].apply(lambda c: c.fillna(row_mean_interp))
    df[cols] = df[cols].interpolate(method='linear', axis=0, limit_direction='both')
    return df

In [ ]:
# ── 1안: PCHIP 보간 ──────────────────────────────────────────────
def _pchip_series(s: pd.Series) -> pd.Series:
    """단일 시계열 PCHIP 보간. 유효값 2개 미만이면 선형으로 fallback."""
    valid = s.notna()
    if valid.sum() < 2:
        return s.interpolate(method='linear', limit_direction='both')
    idx = np.arange(len(s))
    interp_fn = PchipInterpolator(idx[valid], s[valid], extrapolate=True)
    result = s.copy()
    result[~valid] = interp_fn(idx[~valid]).clip(min=0)  # 음수 방지
    return result

def interp_pchip(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    df = df.copy()
    df[cols] = df[cols].apply(_pchip_series)
    return df

In [ ]:
# ── 2안: STL 분해 후 재합성 ──────────────────────────────────────
def _stl_series(s: pd.Series, period: int = STL_PERIOD) -> pd.Series:
    """
    1) 선형 보간으로 STL 입력값 준비
    2) STL 분해 (robust=True: 이상치 내성)
    3) 결측 위치만 trend+seasonal 값으로 채움
    """
    if s.isna().sum() == 0:
        return s
    s_linear = s.interpolate(method='linear', limit_direction='both')
    try:
        res = STL(s_linear, period=period, robust=True).fit()
        reconstructed = pd.Series(res.trend + res.seasonal, index=s.index)
    except Exception:
        return s_linear   # STL 실패 시 선형 fallback
    result = s.copy()
    result[s.isna()] = reconstructed[s.isna()].clip(lower=0)
    return result

def interp_stl(df: pd.DataFrame, cols: list, period: int = STL_PERIOD) -> pd.DataFrame:
    df = df.copy()
    df[cols] = df[cols].apply(lambda c: _stl_series(c, period))
    return df

## 04. 평가 함수 정의

In [ ]:
def rmse(true: np.ndarray, pred: np.ndarray) -> float:
    return np.sqrt(mean_squared_error(true, pred))

def roughness(series: pd.Series) -> float:
    """2차 차분 절댓값 평균 — 낮을수록 매끄러움"""
    return series.diff().diff().abs().mean()

def acf_divergence(s_orig: pd.Series, s_interp: pd.Series,
                   nlags: int = STL_PERIOD) -> float:
    """ACF 곡선 간 RMSE — 낮을수록 시간 구조 보존"""
    a1 = acf(s_orig.dropna(),   nlags=nlags, fft=True)
    a2 = acf(s_interp.dropna(), nlags=nlags, fft=True)
    return np.sqrt(np.mean((a1 - a2) ** 2))

def wass_dist(s_orig: pd.Series, s_interp: pd.Series) -> float:
    """Wasserstein 거리 — 낮을수록 분포 보존"""
    return wasserstein_distance(s_orig.dropna().values,
                                s_interp.dropna().values)

In [ ]:
def make_masks(n: int, length: int, seed: int):
    """
    두 유형의 결측 마스크 생성.
    - random : 전체의 10% 무작위 제거
    - block  : 6H(=36 포인트) 연속 블록 1개 무작위 삽입
    반환: (random_mask_indices, block_mask_indices)
    """
    rng = np.random.default_rng(seed)
    rand_idx  = rng.choice(length, size=int(length * 0.10), replace=False)
    block_start = rng.integers(0, length - 36)
    block_idx   = np.arange(block_start, block_start + 36)
    return rand_idx, block_idx

## 05. 실험 실행
각 세대 × N_TRIALS 반복 × 마스크 유형 2종 → 지표 평균

In [ ]:
methods = {
    '기존안 (대치+선형)': interp_baseline,
    '1안 (PCHIP)':      interp_pchip,
    '2안 (STL)':        interp_stl,
}

records = []

for house in sample_houses:
    s_full = df_eval[house].copy()

    for trial in range(N_TRIALS):
        rand_idx, block_idx = make_masks(
            n=1, length=len(s_full), seed=RANDOM_SEED + trial)

        for mask_type, mask_idx in [('random', rand_idx), ('block', block_idx)]:
            s_masked = s_full.copy()
            s_masked.iloc[mask_idx] = np.nan

            df_tmp = df_eval[['METER_DATE', house]].copy()
            df_tmp[house] = s_masked

            for method_name, method_fn in methods.items():
                df_filled = method_fn(df_tmp, [house])
                s_filled  = df_filled[house]

                records.append({
                    'house':      house,
                    'trial':      trial,
                    'mask_type':  mask_type,
                    'method':     method_name,
                    'rmse':       rmse(s_full.iloc[mask_idx].values,
                                       s_filled.iloc[mask_idx].values),
                    'roughness':  roughness(s_filled),
                    'acf_div':    acf_divergence(s_full, s_filled),
                    'wass_dist':  wass_dist(s_full, s_filled),
                })

df_results = pd.DataFrame(records)
print(f'실험 완료: {len(df_results):,} rows')
df_results.head()

## 06. 결과 집계 및 종합 스코어

In [ ]:
## 지표별 평균 집계 (method × mask_type)
summary = (
    df_results
    .groupby(['method', 'mask_type'])[['rmse','roughness','acf_div','wass_dist']]
    .mean()
    .reset_index()
)

## Z-score 정규화
for col in ['rmse', 'roughness', 'acf_div', 'wass_dist']:
    mu, sigma = summary[col].mean(), summary[col].std()
    summary[col + '_z'] = (summary[col] - mu) / (sigma + 1e-9)

## 가중 합산 → composite score (낮을수록 우수)
W = {'rmse_z': 0.50, 'roughness_z': 0.20, 'acf_div_z': 0.20, 'wass_dist_z': 0.10}
summary['score'] = sum(summary[k] * v for k, v in W.items())

summary_sorted = summary.sort_values(['mask_type', 'score'])
print(summary_sorted[['method','mask_type','rmse','roughness','acf_div','wass_dist','score']]
      .to_string(index=False))

## 07. 시각화

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
metrics = [('rmse', 'RMSE (↓)'), ('roughness', 'Roughness (↓)'),
           ('acf_div', 'ACF Divergence (↓)'), ('wass_dist', 'Wasserstein (↓)')]

for ax, (metric, title) in zip(axes, metrics):
    pivot = summary.pivot(index='method', columns='mask_type', values=metric)
    pivot.plot(kind='bar', ax=ax, rot=15)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('')
    ax.legend(title='mask type', fontsize=9)

plt.suptitle(f'보간법 비교 ({str_domain}, n={len(sample_houses)} 세대 × {N_TRIALS} trials)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'../results/interp_eval_{str_domain}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## 보간 곡선 시각화 — 임의 세대 1개, block mask 1구간
house_ex = sample_houses[0]
s_full   = df_eval[house_ex]
_, block_idx = make_masks(1, len(s_full), RANDOM_SEED)

s_masked = s_full.copy(); s_masked.iloc[block_idx] = np.nan
df_tmp = df_eval[['METER_DATE', house_ex]].copy(); df_tmp[house_ex] = s_masked

fig, ax = plt.subplots(figsize=(16, 4))
view = slice(max(0, block_idx[0] - 30), block_idx[-1] + 30)

ax.plot(s_full.iloc[view].values, 'k-',  lw=1.5, label='원본', alpha=0.7)
styles = {'기존안 (대치+선형)': ('b--',0.8), '1안 (PCHIP)': ('r-',0.9), '2안 (STL)': ('g-.',0.9)}
for name, fn in methods.items():
    s_f = fn(df_tmp, [house_ex])[house_ex]
    ls, alpha = styles[name]
    ax.plot(s_f.iloc[view].values, ls, lw=1.2, alpha=alpha, label=name)

ax.axvspan(30, 30 + len(block_idx), alpha=0.1, color='gray', label='마스킹 구간')
ax.set_title(f'보간 곡선 비교 — {house_ex} (6H block mask)')
ax.legend(); ax.set_xlabel('10분 스텝'); ax.set_ylabel('INST')
plt.tight_layout(); plt.show()

## 08. 결론 및 선정 방법

In [ ]:
best = summary.groupby('method')['score'].mean().idxmin()
score_table = summary.groupby('method')['score'].mean().sort_values()

print('=' * 50)
print('  보간법 종합 순위 (composite score, 낮을수록 우수)')
print('=' * 50)
for rank, (method, score) in enumerate(score_table.items(), 1):
    marker = ' ◀ 선정' if method == best else ''
    print(f'  {rank}위  {method:<25}  score={score:+.4f}{marker}')
print()
print(f'  → data_01_pipeline 에 적용할 보간법: [{best}]')